# 🐱 LongCat-Video-Avatar 1.5/1.6 (美团次世代旗舰数字人)

美团自研 **LongCat-Video (13.6B)** 原生数字人框架，搭载 **OpenAI Whisper-Large-v3** 音频声学编码器与 **DMD2 8 步蒸馏加速**。

与 **Wan2.1 + InfiniteTalk** 相比：
- 🎙️ **更细腻的歌唱咬字**：Whisper-Large-v3 对微弱换气、转音、情感起伏的捕捉远超传统 Wav2Vec2；
- ⚡ **超低步数极速推理**：原生 8 步 DMD2 蒸馏（生成 10 秒约 1 分钟）；
- 🛡️ **强化学习防畸变**：针对手部畸变与长时序漂移做过专门对齐。

In [ ]:
# 1. 检查显卡配置 (推荐 A100 或 L4，显存 >= 16GB 即可)
!nvidia-smi


In [ ]:
# 2. 克隆 LongCat-Video 官方仓库并安装核心依赖
!git clone https://github.com/meituan-longcat/LongCat-Video /content/LongCat-Video || true
%cd /content/LongCat-Video

!pip install -q ninja psutil packaging
!pip install -q flash_attn==2.7.4.post1 --no-build-isolation || true
!pip install -q -r requirements.txt
!pip install -q -r requirements_avatar.txt
!pip install -q "huggingface_hub[cli]"
print("✅ 基础环境安装完成！")


In [ ]:
# 3. 下载模型权重 (LongCat-Video 基座 + LongCat-Video-Avatar 1.5 权重)
!mkdir -p ./weights

print("📥 正在高速下载 LongCat-Video 基础底座 (约 2~3 分钟)... 请稍候")
!huggingface-cli download meituan-longcat/LongCat-Video --local-dir ./weights/LongCat-Video

print("📥 正在下载 LongCat-Video-Avatar-1.5 数字人蒸馏权重...")
!huggingface-cli download meituan-longcat/LongCat-Video-Avatar-1.5 --local-dir ./weights/LongCat-Video-Avatar-1.5

print("🎉 所有模型权重准备就绪！")


In [ ]:
# 4. 准备对齐评测素材 (与 Wan2.1 + InfiniteTalk 使用完全相同的 30 秒歌声与人像)
import json, os, urllib.request

# 下载与当前项目完全一致的对比素材
!curl -s -L "https://raw.githubusercontent.com/ajaxsys/colab-lipsync/main/sample/girl_9x16.jpeg" -o /content/girl_9x16.jpeg
!curl -s -L "https://raw.githubusercontent.com/ajaxsys/colab-lipsync/main/sample/audio_cut_30s.mp3" -o /content/audio_cut_30s.mp3

job_config = {
  "audio_path": "/content/audio_cut_30s.mp3",
  "image_path": "/content/girl_9x16.jpeg",
  "text_prompt": "A beautiful young woman singing emotionally, highly detailed face, realistic skin texture, clear eyes and mouth movements, natural studio lighting.",
  "output_path": "/content/output_longcat_avatar_30s.mp4"
}

with open("/content/job_longcat.json", "w", encoding="utf-8") as f:
    json.dump(job_config, f, ensure_ascii=False, indent=2)

print("✅ 评测输入任务配置已生成：/content/job_longcat.json")


In [ ]:
# 5. 执行 LongCat-Video-Avatar 1.5 极速推理 (单卡 8-Step DMD2 蒸馏)
%cd /content/LongCat-Video

# 如果显存充足 (A100-40GB/80GB)，去掉 --use_int8；如果显存较紧 (T4/L4/RTX4090)，带上 --use_int8
!torchrun --nproc_per_node=1 run_demo_avatar_single_audio_to_video.py \
    --checkpoint_dir=./weights/LongCat-Video-Avatar-1.5 \
    --stage_1=ai2v \
    --input_json=/content/job_longcat.json \
    --use_distill \
    --model_type=avatar-v1.5

print("🎉 LongCat-Video-Avatar 1.5 生成完成！输出路径：/content/output_longcat_avatar_30s.mp4")


In [ ]:
# 6. 播放生成的对比视频
from IPython.display import HTML
from base64 import b64encode

mp4 = open('/content/output_longcat_avatar_30s.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f"""
<video width=400 controls autoplay loop>
      <source src="{data_url}" type="video/mp4">
</video>
""")
